# Obtaining and resampling cryptocurrency data

In [1]:
import requests
import pandas as pd

##Query using the Kraken API, in this case Ethereum data
endpoint = 'https://api.kraken.com/0/public/Trades'
payLoad = {'pair': 'XETHZUSD'}
response = requests.get(url=endpoint, params=payLoad)
tradeData = response.json()
trades = tradeData['result']['XETHZUSD']
#View the top 5 primeras operations
trades[:5]

[['2322.59000', '2.15276400', 1773777673.7146795, 's', 'l', '', 61536090],
 ['2322.46000', '0.05440747', 1773777684.2714686, 'b', 'l', '', 61536091],
 ['2322.27000', '0.00161578', 1773777689.1050766, 's', 'l', '', 61536092],
 ['2322.45000', '0.02174419', 1773777700.227366, 'b', 'l', '', 61536093],
 ['2321.83000', '0.00430694', 1773777723.9597647, 's', 'l', '', 61536094]]

In [2]:
#Create the dataframe,for default the last  1000 operations
tradesDF = pd.DataFrame.from_records(trades,                                    
columns=['Price','Volume','Time','BuySell','MarketLimit','Misc','DUMMY'])
tradesDF = tradesDF.drop(columns=['DUMMY']) 
tradesDF

,Price,Volume,Time,BuySell,MarketLimit,Misc
0,2322.59000,2.15276400,1.773778e+09,s,l,
1,2322.46000,0.05440747,1.773778e+09,b,l,
2,2322.27000,0.00161578,1.773778e+09,s,l,
3,2322.45000,0.02174419,1.773778e+09,b,l,
4,2321.83000,0.00430694,1.773778e+09,s,l,
...,...,...,...,...,...,...
995,2329.35000,0.00420894,1.773781e+09,b,l,
996,2329.35000,0.00481869,1.773781e+09,b,l,
997,2329.35000,0.00292931,1.773781e+09,b,l,
998,2329.33000,0.00257600,1.773781e+09,b,m,


In [4]:
#Create the time index with format date and hour
tradesDF['Time'] = pd.to_datetime(tradesDF['Time'], unit='s')
tradesDF.set_index('Time',inplace=True)
tradesDF

,Price,Volume,BuySell,MarketLimit,Misc
Time,,,,,
2026-03-17 20:01:13.714679480,2322.59000,2.15276400,s,l,
2026-03-17 20:01:24.271468639,2322.46000,0.05440747,b,l,
2026-03-17 20:01:29.105076551,2322.27000,0.00161578,s,l,
2026-03-17 20:01:40.227365971,2322.45000,0.02174419,b,l,
2026-03-17 20:02:03.959764719,2321.83000,0.00430694,s,l,
...,...,...,...,...,...
2026-03-17 21:00:54.773605108,2329.35000,0.00420894,b,l,
2026-03-17 21:00:58.877400160,2329.35000,0.00481869,b,l,
2026-03-17 21:01:00.639366150,2329.35000,0.00292931,b,l,


In [6]:
#query the lastmark in the time
tradeData["result"]["last"]

'1773781280471521680'

In [5]:
pd.to_datetime(int(tradeData["result"]["last"]),unit="ns")

Timestamp('2026-03-17 19:32:58.670274466')

In [8]:
#Resampling per minutes
tradesDF.resample('1min')['Price'].agg(['first'])

,first
Time,
2026-03-17 20:01:00,2322.59000
2026-03-17 20:02:00,2321.83000
2026-03-17 20:03:00,2322.16000
2026-03-17 20:04:00,2324.06000
2026-03-17 20:05:00,2321.80000
...,...
2026-03-17 20:57:00,2328.89000
2026-03-17 20:58:00,2328.72000
2026-03-17 20:59:00,2329.35000


In [9]:
#Obtain the ohlc: Open ,High,Low y Close
ohlc = tradesDF.resample("1min", label="left")["Price"].agg(["first","max","min","last"])
ohlc.columns = ["Open","High","Low","Close"]

In [10]:
ohlc

,Open,High,Low,Close
Time,,,,
2026-03-17 20:01:00,2322.59000,2322.59000,2322.27000,2322.45000
2026-03-17 20:02:00,2321.83000,2322.69000,2321.70000,2322.69000
2026-03-17 20:03:00,2322.16000,2324.54000,2322.16000,2324.46000
2026-03-17 20:04:00,2324.06000,2324.27000,2321.98000,2321.98000
2026-03-17 20:05:00,2321.80000,2321.80000,2319.38000,2321.27000
...,...,...,...,...
2026-03-17 20:57:00,2328.89000,2328.89000,2328.71000,2328.72000
2026-03-17 20:58:00,2328.72000,2329.75000,2328.71000,2329.71000
2026-03-17 20:59:00,2329.35000,2329.36000,2328.85000,2328.89000


## Example with function

In [12]:

import requests
import pandas as pd
import datetime
from datetime import timezone
import time

def getKrakenTradeData(pair, startDate, endDate):
    endpoint = 'https://api.kraken.com/0/public/Trades'
    
    startTime = int(datetime.datetime.strptime(startDate, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp()*1000000000)
    endTime =   int(datetime.datetime.strptime(endDate, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp()*1000000000)
    
    timeLoaded = startTime
    
    result = pd.DataFrame()
    
    while timeLoaded < endTime:
        print(pd.to_datetime(timeLoaded, unit='ns').strftime('%Y-%m-%d %H:%M:%S'))
        payLoad = {'pair': pair,
                   'since': timeLoaded}
        
        response = requests.get(url=endpoint, params=payLoad)
        data = response.json()['result']
        tradesRaw = data[pair]
        timeLoaded = int(data["last"])
        
        tradeData = pd.DataFrame.from_records(tradesRaw,
                             columns=['Price', 'Volume', 'Time', 'BuySell', 'MarketLimit', 'Misc','DUMMY'])
        tradeData['Time'] = pd.to_datetime(tradeData['Time'], unit='s')
        
        result = pd.concat([result, tradeData])
        
        time.sleep(5)
        
    result.set_index("Time", inplace = True)
    result = result.loc[startDate:endDate+' 00:00:00']
    
    return result

### Using the function ,in this case bitcoin with specific dates

In [14]:
df = getKrakenTradeData(
        pair="XXBTZUSD",
        startDate="2026-03-11",
        endDate="2026-03-12"
     )
df = df.drop(columns=['DUMMY']) 
df.tail(10)

2026-03-11 00:00:00
2026-03-11 00:17:24
2026-03-11 00:40:02
2026-03-11 01:14:48
2026-03-11 01:47:45
2026-03-11 02:23:24
2026-03-11 02:49:04
2026-03-11 03:21:21
2026-03-11 04:00:56
2026-03-11 04:33:50
2026-03-11 05:17:42
2026-03-11 05:46:27
2026-03-11 06:21:11
2026-03-11 07:05:10
2026-03-11 07:50:50
2026-03-11 08:35:01
2026-03-11 09:04:57
2026-03-11 09:40:13
2026-03-11 10:14:57
2026-03-11 11:00:44
2026-03-11 11:39:13
2026-03-11 11:59:51
2026-03-11 12:12:46
2026-03-11 12:30:12
2026-03-11 12:49:39
2026-03-11 13:13:57
2026-03-11 13:20:07
2026-03-11 13:31:12
2026-03-11 13:38:02
2026-03-11 13:45:07
2026-03-11 13:52:57
2026-03-11 14:00:59
2026-03-11 14:07:17
2026-03-11 14:15:42
2026-03-11 14:27:24
2026-03-11 14:38:44
2026-03-11 14:51:38
2026-03-11 15:06:06
2026-03-11 15:17:09
2026-03-11 15:31:10
2026-03-11 15:45:03
2026-03-11 16:01:03
2026-03-11 16:11:55
2026-03-11 16:27:40
2026-03-11 16:44:40
2026-03-11 17:01:07
2026-03-11 17:13:50
2026-03-11 17:23:42
2026-03-11 17:30:49
2026-03-11 17:39:29


,Price,Volume,BuySell,MarketLimit,Misc
Time,,,,,
2026-03-11 23:59:34.932181358,70200.40000,0.00013864,b,l,
2026-03-11 23:59:50.419239759,70200.30000,0.00056000,s,l,
2026-03-11 23:59:55.885238409,70200.40000,0.00021156,b,l,
2026-03-12 00:00:00.093214273,70200.40000,0.00064980,b,l,
2026-03-12 00:00:00.093214273,70200.40000,0.00005192,b,l,
2026-03-12 00:00:00.124227524,70200.40000,0.00013964,b,l,
2026-03-12 00:00:00.148654461,70200.40000,0.00021262,b,l,
2026-03-12 00:00:00.159912825,70200.40000,0.00070519,b,l,
2026-03-12 00:00:00.343577147,70200.40000,0.00035086,b,l,


In [15]:
#Resampling in hours
df.resample('1h')['Price'].agg(['first'])

,first
Time,
2026-03-11 00:00:00,69955.40000
2026-03-11 01:00:00,70052.10000
2026-03-11 02:00:00,70084.50000
2026-03-11 03:00:00,69812.00000
2026-03-11 04:00:00,69567.70000
2026-03-11 05:00:00,70133.20000
2026-03-11 06:00:00,69512.20000
2026-03-11 07:00:00,69955.00000
2026-03-11 08:00:00,69647.10000


In [16]:
# Obtain the OHLC data: Open,High,Low,Close
ohlc = df.resample("1h", label="left")["Price"].agg(["first","max","min","last"])
ohlc.columns = ["Open","High","Low","Close"]
ohlc

,Open,High,Low,Close
Time,,,,
2026-03-11 00:00:00,69955.40000,70165.20000,69782.10000,70052.10000
2026-03-11 01:00:00,70052.10000,70138.60000,69902.10000,70084.40000
2026-03-11 02:00:00,70084.50000,70121.90000,69644.80000,69803.70000
2026-03-11 03:00:00,69812.00000,69887.00000,69517.10000,69567.60000
2026-03-11 04:00:00,69567.70000,70251.80000,69519.80000,70154.50000
2026-03-11 05:00:00,70133.20000,70186.90000,69450.00000,69512.10000
2026-03-11 06:00:00,69512.20000,69985.30000,69512.10000,69955.00000
2026-03-11 07:00:00,69955.00000,70012.40000,69512.40000,69649.20000
2026-03-11 08:00:00,69647.10000,69793.80000,69512.10000,69665.10000
